## Ablation Study: Removing PCA

This notebook repeats the pipeline from `HSI_Final_Code.ipynb` **without the PCA step** and compares the results with the original. The original notebook and its outputs are not changed.

In the original pipeline, PCA turns each hyperspectral cube into a 3-channel image. That image is used in two places:

1. As the input to **SLIC** when making the pseudo labels.
2. As the input image for **SegFormer**.

Removing PCA affects both places, so this notebook runs two experiments:

| Model | Model input | Trained on |
|---|---|---|
| **Original** (with PCA) | 3 PCA components | PCA pseudo labels |
| **A**: no PCA in model input | all spectral bands | PCA pseudo labels (same as original) |
| **B**: no PCA anywhere | all spectral bands | no-PCA pseudo labels |

All three models are scored on the **same validation images**, against **both** sets of pseudo labels.

**Before running:** the raw dataset must be in the `Dataset/` folder, and the original notebook must have been run once so that `TrainData/` and `SegformerResults/final_model/` exist.

In [ ]:
import os, glob, json, re, time, warnings
import numpy as np
import pandas as pd
import scipy.io as sio
import cv2
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from skimage.segmentation import slic
from skimage.morphology import opening, closing, disk
from skimage.measure import label, regionprops

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation, TrainingArguments, Trainer

warnings.filterwarnings("ignore")

## 0. Settings

In [ ]:
# ---- Raw data (same as the original notebook) ----
DATASET_DIR = 'Dataset'
SUBFOLDERS = ['DNCube', 'MMCube', 'OtherCube']

# ---- Outputs of the ORIGINAL notebook (only read, never changed) ----
ORIG_IMAGE_DIR = 'TrainData/images'
ORIG_MASK_DIR = 'TrainData/masks'
ORIG_RESULTS_DIR = 'SegformerResults'
ORIG_MODEL_DIR = os.path.join(ORIG_RESULTS_DIR, 'final_model')

# ---- Outputs of THIS notebook ----
CUBE_DIR = 'NoPCA_TrainData/cubes'        # full-band model inputs (.npy)
NOPCA_MASK_DIR = 'NoPCA_TrainData/masks'  # pseudo labels made without PCA
RESULTS_DIR = 'NoPCA_Results'             # trained no-PCA models
COMPARISON_DIR = 'Comparison'             # tables and charts

# ---- Training settings, identical to the original notebook ----
PRETRAINED = 'nvidia/segformer-b0-finetuned-ade-512-512'
IMG_SIZE = 256
EPOCHS = 20
BATCH_SIZE = 2
LEARNING_RATE = 5e-5
SEED = 42

# ---- Settings for this notebook ----
SLIC_COMPACTNESS_NO_PCA = 1   # the original used 10 on the PCA image; see the note in Section 2
RUN_EXPERIMENT_A = True
RUN_EXPERIMENT_B = True
REUSE_TRAINED_MODELS = True   # if a model was already trained, load it instead of training again

for d in [CUBE_DIR, NOPCA_MASK_DIR, RESULTS_DIR, COMPARISON_DIR]:
    os.makedirs(d, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Functions from the original notebook (same logic)

In [ ]:
def normalize_hsi_cube(hsi):
    # Normalize each band individually to [0, 1] range.
    hsi_norm = np.zeros_like(hsi)
    for i in range(hsi.shape[2]):
        band = hsi[:, :, i]
        band_min, band_max = np.min(band), np.max(band)
        hsi_norm[:, :, i] = (band - band_min) / (band_max - band_min + 1e-8)
    return hsi_norm


def refine_mask(mask):
    # Keep only the region closest to the image centre.
    labeled = label(mask)
    regions = regionprops(labeled)
    if not regions:
        return mask
    h, w = mask.shape
    center = np.array([h / 2, w / 2])
    selected_region = min(regions, key=lambda r: np.linalg.norm(center - np.array(r.centroid)))
    refined_mask = np.zeros_like(mask)
    refined_mask[labeled == selected_region.label] = 1
    return refined_mask


def apply_elliptical_roi(shape, scale=0.9):
    # Elliptical mask covering the central region of the image.
    h, w = shape[:2]
    y, x = np.ogrid[:h, :w]
    cy, cx = h // 2, w // 2
    ry, rx = h * scale / 2, w * scale / 2
    roi = ((y - cy) ** 2 / ry ** 2 + (x - cx) ** 2 / rx ** 2) <= 1
    return roi.astype(np.uint8)


def find_cube_files(dataset_dir, subfolders):
    # List the .mat files in the same folder order as the original notebook.
    # If two folders hold the same file name, the later folder wins,
    # which matches how the original notebook overwrote its saved PNGs.
    files, duplicates = {}, 0
    for folder in subfolders:
        folder_path = os.path.join(dataset_dir, folder)
        if not os.path.isdir(folder_path):
            raise FileNotFoundError(f"Missing folder: {folder_path}. Put the raw dataset in '{dataset_dir}/' first.")
        for file in sorted(os.listdir(folder_path)):
            if file.endswith('.mat'):
                name = file[:-4]
                duplicates += name in files
                files[name] = os.path.join(folder_path, file)
    if duplicates:
        print(f"Note: {duplicates} file names appear in more than one folder; the later folder was used.")
    return files


def load_cube(path):
    # Return the first 3-D array in a .mat file (same rule as the original loader).
    mat = sio.loadmat(path)
    for key in mat:
        if isinstance(mat[key], np.ndarray) and mat[key].ndim == 3:
            return mat[key]
    raise ValueError(f"No 3-D array found in {path}")

## 2. The one change: SLIC without PCA

This is the original `generate_mask` with one difference: SLIC runs on **all spectral bands** instead of the 3 PCA components, with a compactness suited to that input. The KMeans step already used all bands in the original, so it is unchanged.

**Note on compactness.** Compactness balances colour similarity against distance. scikit-image first rescales any SLIC input to the range 0 to 1. For the original 3-channel PCA image it then converts to Lab colour, where values run from 0 to 100, and `compactness=10` is balanced for that. With all bands there is no Lab conversion, so colour differences are about 100 times smaller. Keeping 10 would make SLIC ignore colour and cut the image into square blocks.

This notebook therefore uses `SLIC_COMPACTNESS_NO_PCA = 1`. In a test on synthetic cubes, values from 0.3 to 1 followed the lesion edges, 10 produced square blocks, and 0.1 or lower broke apart. If the no-PCA labels in Section 9 still look blocky, lower it. If they look ragged or broken, raise it.

In [ ]:
def generate_mask_no_pca(hsi_cube, n_clusters=2):
    # SLIC superpixels on the full normalized cube (no PCA)
    segments = slic(hsi_cube, n_segments=200, compactness=SLIC_COMPACTNESS_NO_PCA,
                    start_label=0, channel_axis=-1, convert2lab=False)
    h, w = segments.shape
    superpixel_labels = np.unique(segments)

    # Mean spectral signature of each superpixel (unchanged)
    signatures = np.array([hsi_cube[segments == s].mean(axis=0) for s in superpixel_labels])

    # KMeans clustering (unchanged)
    cluster_labels = KMeans(n_clusters=n_clusters, random_state=42).fit_predict(signatures)

    # Superpixel clusters to pixel-wise mask (unchanged)
    mask = np.zeros((h, w), dtype=np.uint8)
    for i, s in enumerate(superpixel_labels):
        mask[segments == s] = cluster_labels[i]

    # Lesion = smaller cluster (unchanged)
    lesion_mask = (mask == 1).astype(np.uint8) if np.sum(mask == 1) < np.sum(mask == 0) else (mask == 0).astype(np.uint8)

    # Morphological filtering (unchanged)
    lesion_mask = opening(lesion_mask, disk(3))
    lesion_mask = closing(lesion_mask, disk(5))
    return lesion_mask

## 3. Build the no-PCA data

For every cube this saves the full-band input as a `.npy` file and the no-PCA pseudo label as a `.png` file. Files that already exist are skipped, so this cell can be re-run safely.

In [ ]:
cube_files = find_cube_files(DATASET_DIR, SUBFOLDERS)
print(f"Found {len(cube_files)} cubes in {DATASET_DIR}/")

start = time.time()
made = 0
for i, (name, path) in enumerate(sorted(cube_files.items())):
    cube_path = os.path.join(CUBE_DIR, name + '.npy')
    mask_path = os.path.join(NOPCA_MASK_DIR, name + '.png')
    if os.path.exists(cube_path) and os.path.exists(mask_path):
        continue

    hsi = np.nan_to_num(load_cube(path).astype(np.float64))
    hsi = normalize_hsi_cube(hsi)

    mask = generate_mask_no_pca(hsi)
    mask = mask * apply_elliptical_roi(mask.shape)
    mask = refine_mask(mask)

    np.save(cube_path, hsi.astype(np.float16))
    Image.fromarray((mask * 255).astype(np.uint8)).save(mask_path)
    made += 1
    if made % 25 == 0:
        print(f"  {made} done ({time.time() - start:.0f} s)")

print(f"Created {made} new files, skipped {len(cube_files) - made} existing ones.")

In [ ]:
# Sanity check: the no-PCA data must line up with the original TrainData
def png_names(folder):
    return {f[:-4] for f in os.listdir(folder) if f.endswith('.png')}

orig_names = png_names(ORIG_IMAGE_DIR) & png_names(ORIG_MASK_DIR)
new_names = {f[:-4] for f in os.listdir(CUBE_DIR) if f.endswith('.npy')} & png_names(NOPCA_MASK_DIR)

# Same ordering as the original notebook, which sorted the file paths
NAMES = sorted(orig_names & new_names, key=lambda n: n + '.png')

N_BANDS = np.load(os.path.join(CUBE_DIR, NAMES[0] + '.npy'), mmap_mode='r').shape[-1]
cube_hw = np.load(os.path.join(CUBE_DIR, NAMES[0] + '.npy'), mmap_mode='r').shape[:2]
png_hw = Image.open(os.path.join(ORIG_IMAGE_DIR, NAMES[0] + '.png')).size[::-1]

print(f"Original images: {len(orig_names)}")
print(f"No-PCA cubes:    {len(new_names)}")
print(f"Used in both:    {len(NAMES)}")
print(f"Spectral bands:  {N_BANDS}  (the original used 3 PCA components)")
print(f"Image size:      cube {cube_hw}, original PNG {png_hw}")

assert cube_hw == png_hw, "Cube and PNG sizes differ, so the files do not belong together."
if len(NAMES) != len(orig_names):
    print("WARNING: some original images have no matching cube. The validation split will differ "
          "from the original, so the original model may have trained on some validation images.")

## 4. Same train/validation split as the original

The original split 80/20 with `random_state=42` over the sorted image list. Using the same list and the same seed gives the same validation images, which the original model never trained on.

In [ ]:
train_idx, val_idx = train_test_split(np.arange(len(NAMES)), test_size=0.2, random_state=SEED)
TRAIN_NAMES = [NAMES[i] for i in train_idx]
VAL_NAMES = [NAMES[i] for i in val_idx]
print(f"Training images: {len(TRAIN_NAMES)}")
print(f"Validation images: {len(VAL_NAMES)}")

# Per-band mean and std from the training images only, used to standardize the no-PCA input
total = np.zeros(N_BANDS); total_sq = np.zeros(N_BANDS); count = 0
for name in TRAIN_NAMES:
    x = np.load(os.path.join(CUBE_DIR, name + '.npy')).astype(np.float64).reshape(-1, N_BANDS)
    total += x.sum(0); total_sq += (x ** 2).sum(0); count += len(x)
BAND_MEAN = (total / count).astype(np.float32)
BAND_STD = (np.sqrt(np.maximum(total_sq / count - BAND_MEAN.astype(np.float64) ** 2, 0)) + 1e-6).astype(np.float32)

## 5. Datasets and model

- **No-PCA input:** all bands, standardized per band. Same resize and flips as the original.
- **Original input:** the saved PCA images, prepared exactly as in the original notebook. Only used to score the original model.
- **Validation images are not flipped**, so every score in this notebook is repeatable.
- **Model:** the pretrained SegFormer expects 3 input channels. For the no-PCA models, its first layer is widened to all bands by averaging its RGB weights, so the rest of the pretrained network is still used.

In [ ]:
def mask_from_png(path):
    return (np.array(Image.open(path).convert('L')) > 128).astype(np.uint8)

train_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=0.3), A.VerticalFlip(p=0.3), ToTensorV2()])
eval_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), ToTensorV2()])
pca_eval_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
                         A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                         ToTensorV2()])


class CubeDataset(Dataset):
    # All spectral bands as input (no PCA).
    def __init__(self, names, mask_dir, transform):
        self.names, self.mask_dir, self.transform = names, mask_dir, transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        x = np.load(os.path.join(CUBE_DIR, name + '.npy')).astype(np.float32)
        x = ((x - BAND_MEAN) / BAND_STD).astype(np.float32)
        y = mask_from_png(os.path.join(self.mask_dir, name + '.png'))
        t = self.transform(image=x, mask=y)
        return {'pixel_values': t['image'].float(), 'labels': t['mask'].long()}


class PCAImageDataset(Dataset):
    # The original 3-channel PCA images, prepared as in the original notebook.
    def __init__(self, names, mask_dir):
        self.names, self.mask_dir = names, mask_dir

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        x = np.array(Image.open(os.path.join(ORIG_IMAGE_DIR, name + '.png')).convert('RGB'))
        y = mask_from_png(os.path.join(self.mask_dir, name + '.png'))
        t = pca_eval_tf(image=x, mask=y)
        return {'pixel_values': t['image'].float(), 'labels': t['mask'].long()}


def collate(batch):
    return {'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
            'labels': torch.stack([b['labels'] for b in batch])}


def build_model(n_channels):
    model = SegformerForSemanticSegmentation.from_pretrained(PRETRAINED, num_labels=2, ignore_mismatched_sizes=True)
    if n_channels == 3:
        return model
    old = model.segformer.encoder.patch_embeddings[0].proj
    new = torch.nn.Conv2d(n_channels, old.out_channels, kernel_size=old.kernel_size,
                          stride=old.stride, padding=old.padding)
    with torch.no_grad():
        new.weight.copy_(old.weight.mean(dim=1, keepdim=True).repeat(1, n_channels, 1, 1) * 3 / n_channels)
        new.bias.copy_(old.bias)
    model.segformer.encoder.patch_embeddings[0].proj = new
    model.config.num_channels = n_channels
    return model


sample = CubeDataset(TRAIN_NAMES, ORIG_MASK_DIR, train_tf)[0]
print('No-PCA input shape:', tuple(sample['pixel_values'].shape), '| label shape:', tuple(sample['labels'].shape))

## 6. Train the no-PCA models

Each experiment trains with the same settings as the original: 20 epochs, batch size 2, learning rate 5e-5, and the best epoch chosen by validation Dice. **Each one takes about as long as the original training.**

If a model was already trained, it is loaded instead of trained again. Set `REUSE_TRAINED_MODELS = False` in Section 0 to force retraining.

In [ ]:
def scores(pred, true):
    # Pixel metrics for binary masks, lesion = 1.
    pred = np.asarray(pred).astype(bool).ravel()
    true = np.asarray(true).astype(bool).ravel()
    tp = np.sum(pred & true); fp = np.sum(pred & ~true)
    fn = np.sum(~pred & true); tn = np.sum(~pred & ~true)
    div = lambda a, b: float(a / b) if b > 0 else 0.0
    return {'accuracy': div(tp + tn, tp + tn + fp + fn),
            'precision': div(tp, tp + fp),
            'recall': div(tp, tp + fn),
            'specificity': div(tn, tn + fp),
            'dice': div(2 * tp, 2 * tp + fp + fn),
            'iou': div(tp, tp + fp + fn)}


def upsample(logits, size):
    # Class with the highest score, resized to the label size (nearest neighbour, as in the original).
    pred = logits.argmax(1, keepdim=True).float()
    return F.interpolate(pred, size=size, mode='nearest').squeeze(1)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    pred = upsample(torch.from_numpy(logits), labels.shape[-2:]).numpy()
    s = scores(pred, labels)
    return {'f1_score': s['dice'], 'iou': s['iou'], 'accuracy': s['accuracy']}


def train_experiment(tag, mask_dir):
    out_dir = os.path.join(RESULTS_DIR, tag)
    final_dir = os.path.join(out_dir, 'final_model')
    if REUSE_TRAINED_MODELS and os.path.exists(os.path.join(final_dir, 'config.json')):
        print(f"{tag}: found a trained model, loading it instead of training again.")
        return SegformerForSemanticSegmentation.from_pretrained(final_dir).to(device)

    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_steps=5,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model='eval_f1_score',
        greater_is_better=True,
        report_to=[],
        save_total_limit=2,
        dataloader_pin_memory=False,
        seed=SEED,
    )
    trainer = Trainer(
        model=build_model(N_BANDS),
        args=args,
        train_dataset=CubeDataset(TRAIN_NAMES, mask_dir, train_tf),
        eval_dataset=CubeDataset(VAL_NAMES, mask_dir, eval_tf),
        compute_metrics=compute_metrics,
        data_collator=collate,
    )
    start = time.time()
    trainer.train()
    print(f"{tag}: training took {(time.time() - start) / 60:.1f} minutes")
    trainer.save_model(final_dir)
    with open(os.path.join(out_dir, 'log_history.json'), 'w') as f:
        json.dump(trainer.state.log_history, f, indent=1)
    return trainer.model.to(device)

In [ ]:
# Experiment A: no PCA in the model input, trained on the ORIGINAL (PCA) pseudo labels
model_a = train_experiment('exp_a_no_pca_input', ORIG_MASK_DIR) if RUN_EXPERIMENT_A else None

In [ ]:
# Experiment B: no PCA anywhere, trained on the NO-PCA pseudo labels
model_b = train_experiment('exp_b_no_pca_anywhere', NOPCA_MASK_DIR) if RUN_EXPERIMENT_B else None

## 7. Score all models on the same validation images

Every model is scored against both sets of pseudo labels. Specificity here is the true specificity, TN / (TN + FP), for all models.

In [ ]:
@torch.no_grad()
def predict(model, dataset):
    model.eval().to(device)
    preds = []
    for batch in DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate):
        logits = model(pixel_values=batch['pixel_values'].to(device)).logits
        preds.append(upsample(logits, batch['labels'].shape[-2:]).cpu().numpy().astype(np.uint8))
    return np.concatenate(preds)


def val_labels(mask_dir):
    # Validation labels resized to IMG_SIZE with nearest neighbour, as the model inputs were.
    return np.stack([cv2.resize(mask_from_png(os.path.join(mask_dir, n + '.png')), (IMG_SIZE, IMG_SIZE),
                                interpolation=cv2.INTER_NEAREST) for n in VAL_NAMES])


LABEL_SETS = {'PCA labels': val_labels(ORIG_MASK_DIR), 'No-PCA labels': val_labels(NOPCA_MASK_DIR)}

PREDICTIONS = {}
if os.path.exists(os.path.join(ORIG_MODEL_DIR, 'config.json')):
    orig_model = SegformerForSemanticSegmentation.from_pretrained(ORIG_MODEL_DIR)
    PREDICTIONS['Original (PCA)'] = predict(orig_model, PCAImageDataset(VAL_NAMES, ORIG_MASK_DIR))
else:
    print(f"Original model not found in {ORIG_MODEL_DIR}. Run HSI_Final_Code.ipynb first to include it.")
if model_a is not None:
    PREDICTIONS['A: no PCA input'] = predict(model_a, CubeDataset(VAL_NAMES, ORIG_MASK_DIR, eval_tf))
if model_b is not None:
    PREDICTIONS['B: no PCA anywhere'] = predict(model_b, CubeDataset(VAL_NAMES, NOPCA_MASK_DIR, eval_tf))

rows = [{'Model': m, 'Scored against': l, **scores(pred, LABEL_SETS[l])}
        for m, pred in PREDICTIONS.items() for l in LABEL_SETS]
results = pd.DataFrame(rows)
results.to_csv(os.path.join(COMPARISON_DIR, 'comparison_results.csv'), index=False)
print(f"Saved {os.path.join(COMPARISON_DIR, 'comparison_results.csv')}")
results.round(4)

## 8. How much does removing PCA change the pseudo labels?

This compares the two sets of pseudo labels directly, image by image, with the PCA labels as the reference. A Dice of 1.0 would mean they are identical.

In [ ]:
agreement = pd.DataFrame([{'image': n, **scores(mask_from_png(os.path.join(NOPCA_MASK_DIR, n + '.png')),
                                                mask_from_png(os.path.join(ORIG_MASK_DIR, n + '.png')))}
                          for n in NAMES])
agreement.to_csv(os.path.join(COMPARISON_DIR, 'pseudo_label_agreement.csv'), index=False)

print(f"Pseudo-label agreement over {len(NAMES)} images (no-PCA vs PCA):")
print(f"  Mean Dice: {agreement['dice'].mean():.4f}")
print(f"  Mean IoU:  {agreement['iou'].mean():.4f}")
print(f"  Images with Dice below 0.5: {(agreement['dice'] < 0.5).sum()}")

## 9. Charts

In [ ]:
# Dice and IoU for every model, against both label sets
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric in zip(axes, ['dice', 'iou']):
    table = results.pivot(index='Model', columns='Scored against', values=metric).reindex(list(PREDICTIONS.keys()))
    table.plot(kind='bar', ax=ax, rot=0, color=['#4C72B0', '#DD8452'])
    ax.set_title(f'Validation {metric.upper() if metric == "iou" else metric.title()}')
    ax.set_ylim(0, 1); ax.set_xlabel(''); ax.grid(axis='y', alpha=0.3)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.3f', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(COMPARISON_DIR, 'metrics_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Validation Dice per epoch during training
def read_history(path):
    with open(path) as f:
        return json.load(f)

curves = {}
orig_states = glob.glob(os.path.join(ORIG_RESULTS_DIR, 'checkpoint-*', 'trainer_state.json'))
if orig_states:
    latest = max(orig_states, key=lambda p: int(re.search(r'checkpoint-(\d+)', p).group(1)))
    curves['Original (PCA)'] = read_history(latest)['log_history']
for tag, label_ in [('exp_a_no_pca_input', 'A: no PCA input'), ('exp_b_no_pca_anywhere', 'B: no PCA anywhere')]:
    path = os.path.join(RESULTS_DIR, tag, 'log_history.json')
    if os.path.exists(path):
        curves[label_] = read_history(path)

plt.figure(figsize=(7, 4))
for name, history in curves.items():
    points = [(h['epoch'], h['eval_f1_score']) for h in history if 'eval_f1_score' in h]
    if points:
        plt.plot(*zip(*points), marker='o', markersize=3, label=name)
plt.title('Validation Dice during training (each model vs its own training labels)')
plt.xlabel('Epoch'); plt.ylabel('Dice'); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(COMPARISON_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side examples from the validation set
show = list(range(min(4, len(VAL_NAMES))))
columns = ['PCA image\n(original input)', f'3 of {N_BANDS} bands\n(display only)',
           'PCA pseudo label', 'No-PCA pseudo label'] + [f'Prediction\n{m}' for m in PREDICTIONS]

fig, axes = plt.subplots(len(show), len(columns), figsize=(2.6 * len(columns), 2.2 * len(show)), squeeze=False)
for r, i in enumerate(show):
    name = VAL_NAMES[i]
    cube = np.load(os.path.join(CUBE_DIR, name + '.npy')).astype(np.float32)
    composite = cube[..., [N_BANDS - 1, N_BANDS // 2, 0]]
    panels = [np.array(Image.open(os.path.join(ORIG_IMAGE_DIR, name + '.png')).convert('RGB')),
              (composite - composite.min()) / (np.ptp(composite) + 1e-8),
              mask_from_png(os.path.join(ORIG_MASK_DIR, name + '.png')),
              mask_from_png(os.path.join(NOPCA_MASK_DIR, name + '.png'))] + [p[i] for p in PREDICTIONS.values()]
    for c, panel in enumerate(panels):
        ax = axes[r, c]
        ax.imshow(panel, cmap=None if panel.ndim == 3 else 'gray', aspect='auto')
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(columns[c], fontsize=9)
    axes[r, 0].set_ylabel(name, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(COMPARISON_DIR, 'visual_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary

In [ ]:
def metric(model, labels, name='dice'):
    row = results[(results['Model'] == model) & (results['Scored against'] == labels)]
    return float(row[name].iloc[0]) if len(row) else None

def line(text, a, b):
    if a is None or b is None:
        print(f"{text}: not available (a model was skipped or not found)")
    else:
        print(f"{text}: {a:.4f} -> {b:.4f}  (change {b - a:+.4f})")

print("=" * 70)
print("RESULT SUMMARY  (Dice on the same validation images)")
print("=" * 70)
line("1. Remove PCA from the model input only\n   Original vs A, both scored on PCA labels",
     metric('Original (PCA)', 'PCA labels'), metric('A: no PCA input', 'PCA labels'))
line("2. Remove PCA everywhere\n   Original on PCA labels vs B on no-PCA labels",
     metric('Original (PCA)', 'PCA labels'), metric('B: no PCA anywhere', 'No-PCA labels'))
print(f"3. Pseudo labels with vs without PCA agree with mean Dice {agreement['dice'].mean():.4f}")
print("=" * 70)
print(f"Tables and charts are saved in the '{COMPARISON_DIR}' folder.")

## How to read the results

- **Comparison 1, Original vs A**, is the cleanest test. Both models are trained and scored on the same PCA pseudo labels, and only the input changes. It shows whether SegFormer does better with 3 PCA components or with all bands.
- **Comparison 2, Original vs B**, is the full "no PCA" pipeline. Each model is scored on its own kind of labels, as the original report does. The labels differ, so treat this as indicative, not exact.
- **Pseudo-label agreement** shows how much PCA changes the labels themselves. Low agreement means the two pipelines are learning different targets.
- **None of the labels are manual ground truth.** High scores mean the model reproduces its pseudo labels well, not that it is clinically accurate.
- **The original model's numbers may differ slightly from its own notebook.** Here, validation images are not randomly flipped, and specificity uses the standard formula.